In [ ]:
# Task 1: classify Parkinson's vs control

In [ ]:
import sys, json
from pathlib import Path
import numpy as np
import pandas as pd
import joblib
from sklearn.dummy import DummyClassifier
from sklearn.metrics import accuracy_score, f1_score, roc_auc_score

ROOT = Path.cwd().resolve()
if not (ROOT / "src").is_dir():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))
sys.path.insert(0, str(ROOT / "src"))

# build data/processed/ if it's missing (features.py needs it)
from data import txt_to_csv, raw_dir, out_dir
if not out_dir.exists():
    txt_to_csv(raw_dir, out_dir)

from features import X, y      # X: 165 subjects x 5 features, y: 1 = PD
from src.models.logreg import make_logistic_regression
from src.models.rf import make_random_forest

OUT = ROOT / "presentation_material" / "task1"
(OUT / "predictions").mkdir(parents=True, exist_ok=True)
(OUT / "weights").mkdir(parents=True, exist_ok=True)

with open(ROOT / "data" / "splits.json") as f:
    assignment = json.load(f)["task1"]

test = set(assignment["test"])
folds = []
for fold in assignment["folds"]:
    tr, va = set(fold["train"]), set(fold["val"])
    assert tr.isdisjoint(va | test) and va.isdisjoint(test)
    folds.append((np.flatnonzero(X.index.isin(tr)), np.flatnonzero(X.index.isin(va))))

print(f"{len(X)} subjects | PD share = {y.mean():.3f} | {len(folds)} folds | {len(test)} test subjects held out")

In [ ]:
def train_and_save(model_name, make_model):
    for k, (tr, va) in enumerate(folds, start=1):
        model = make_model()                       # fresh, unfitted model per fold
        model.fit(X.iloc[tr], y.iloc[tr])
        prob = model.predict_proba(X.iloc[va])[:, 1]   # P(PD)
        pd.DataFrame({
            "subject": X.index[va],
            "fold": k,
            "y_true": y.iloc[va].to_numpy(),
            "y_pred": (prob >= 0.5).astype(int),
            "y_prob": prob,
        }).to_csv(OUT / "predictions" / f"{model_name}_val_fold{k}.csv", index=False)
        joblib.dump(model, OUT / "weights" / f"{model_name}_val_fold{k}.joblib")
    print(f"{model_name}: done")

In [ ]:
# "prior" always predicts the training-fold majority class (PD) and gives a constant probability
train_and_save("majority_baseline", lambda: DummyClassifier(strategy="prior"))

In [ ]:
train_and_save("logistic_regression", make_logistic_regression)

In [ ]:
train_and_save("random_forest", lambda: make_random_forest(task="classification"))

In [ ]:
def fold_metrics(name):
    rows = []
    for k in range(1, len(folds) + 1):
        d = pd.read_csv(OUT / "predictions" / f"{name}_val_fold{k}.csv")
        rows.append({"accuracy": accuracy_score(d.y_true, d.y_pred),
                     "f1": f1_score(d.y_true, d.y_pred),
                     "roc_auc": roc_auc_score(d.y_true, d.y_prob)})
    return pd.DataFrame(rows)

summary = {}
for name in ["majority_baseline", "logistic_regression", "random_forest"]:
    m = fold_metrics(name)
    summary[name] = {f"{c}": f"{m[c].mean():.3f} ± {m[c].std():.3f}" for c in m}
pd.DataFrame(summary).T